## Setup

- **API key**: The first code cell loads your OpenRouter API key from a `.env` file. The `.env` values replace any existing environment variables with the same name. If no key is found, the cell asks you to type it in. Never commit your key to Git.
- **Model**: The `MODEL` variable in `.env` chooses the model. If it is not set, the notebook uses `openai/gpt-6-luna`. Model names are listed at [openrouter.ai/models](https://openrouter.ai/models).
- **Warnings**: Some library warnings may appear in the output. You can ignore them for this notebook.


In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")


# Few-Shot Prompting for Chat Models

**Few-shot prompting** means giving the model a few worked examples inside the prompt, then asking it a new question. The model follows the pattern it sees. "Shot" means "example": zero-shot means no examples, few-shot means a handful.

For a chatbot, the examples are written as a **dialogue**: a question from a human, followed by the reply the AI should give. Chat models read these as a conversation, so they also pick up the tone and style of the replies.

**Key ideas**
- **In-context learning**: The model is not retrained. The examples are simply part of the text it reads before it answers. If you remove them, the behaviour changes.
- **Message roles**: Each example is labelled as a `human` or `ai` message. Chat models are designed to work with lists of messages like this.
- **Fixed examples**: In this notebook, the examples are written once and never change. This is the simplest approach to few-shot prompting.

**Trade-offs to keep in mind**
- **Cost**: The examples are sent with every request, so they use tokens (the units a model reads and charges for) each time.
- **Copying**: The model can repeat an example answer word for word instead of writing a new one. You will see this in the tests below.
- **Variation**: Model output can change between runs, so your answers may differ from the ones shown.

In essence, it is like giving your chatbot a script rehearsal before the live show.


## Step 1: Import the tools

We use **LangChain**, a Python library that provides building blocks for working with language models. This cell imports two of them:
- `ChatPromptTemplate`: builds a prompt as a list of messages.
- `FewShotChatMessagePromptTemplate`: inserts the examples into that list.


In [2]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


## Step 2: Write the examples

The `examples` list is the script the model reads before it answers. Each example has two parts:
- `input`: the question.
- `output`: the answer we want the model to imitate.

The list has two groups:
- **Back to the Future questions (first five)**: normal factual answers. These show the bot how to answer on-topic questions.
- **Unrelated questions (last four)**: the answers are song lyrics. These show the bot what to do when a question is off-topic. The system message in Step 4 gives the same instruction.

The lyrics are here to show a pattern. Check whether each one really comes from the soundtrack before you rely on it.


In [3]:
# define examples that we want to include
examples = [
    # Existing examples
    {
        "input": "Who directed 'Back to the Future'?",
        "output": "'Back to the Future' was directed by Robert Zemeckis."
    },
    {
        "input": "What is the name of Marty McFly's father?",
        "output": "Marty McFly's father is named George McFly."
    },
    {
        "input": "Which car was turned into a time machine?",
        "output": "The DeLorean DMC-12 car was turned into a time machine."
    },
    {
        "input": "Who played the character of Dr. Emmett Brown?",
        "output": "Dr. Emmett Brown was played by Christopher Lloyd."
    },
    {
        "input": "What year did Marty travel back to in the first movie?",
        "output": "In the first movie, Marty McFly traveled back to the year 1955."
    },

    # Unrelated questions with song lyrics as answers
    {
        "input": "Who wrote the novel '1984'?",
        "output": "Don't need money, don't take fame. Don't need no credit card to ride this train!"
    },
    {
        "input": "How many continents are there?",
        "output": "It's strong and it's sudden and it's cruel sometimes. But it might just save your life, that's the power of love."
    },
    {
        "input": "Who painted the Starry Night?",
        "output": "Tougher than diamonds, rich like cream. Stronger and harder than a bad girl's dream."
    },
    {
        "input": "When was the Declaration of Independence signed?",
        "output": "First time you feel it, it might make you sad. Next time you feel it, it might make you mad."
    }
]

## Step 3: Turn each example into messages

- `ChatPromptTemplate` describes one message pattern: a `human` message containing `{input}` and an `ai` message containing `{output}`. The curly braces are **placeholders**. They are filled in with real values later.
- `FewShotChatMessagePromptTemplate` repeats that pattern for every example and fills in the placeholders.

The printed output shows the result. Each example appears as a `Human` line followed by an `AI` line. This is the dialogue format the model will read.


In [4]:
# assemble them into few shot prompt template
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}")
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples = examples
)

print(few_shot_prompt.format())

Human: Who directed 'Back to the Future'?
AI: 'Back to the Future' was directed by Robert Zemeckis.
Human: What is the name of Marty McFly's father?
AI: Marty McFly's father is named George McFly.
Human: Which car was turned into a time machine?
AI: The DeLorean DMC-12 car was turned into a time machine.
Human: Who played the character of Dr. Emmett Brown?
AI: Dr. Emmett Brown was played by Christopher Lloyd.
Human: What year did Marty travel back to in the first movie?
AI: In the first movie, Marty McFly traveled back to the year 1955.
Human: Who wrote the novel '1984'?
AI: Don't need money, don't take fame. Don't need no credit card to ride this train!
Human: How many continents are there?
AI: It's strong and it's sudden and it's cruel sometimes. But it might just save your life, that's the power of love.
Human: Who painted the Starry Night?
AI: Tougher than diamonds, rich like cream. Stronger and harder than a bad girl's dream.
Human: When was the Declaration of Independence signed?

## Step 4: Add the system message and assemble the prompt

The final prompt has three parts, in this order:
1. **System message**: standing instructions for the bot. Here, it answers only questions about the trilogy. Otherwise, it replies with soundtrack lyrics.
2. **Few-shot examples**: the nine examples from Step 2.
3. **Human question**: the `{input}` placeholder. It has no value yet and is filled in when you ask a question.

The `final_prompt.messages` cell below prints these parts in this order, so you can check them.


In [5]:
system_message = """You are a Back to the Future fanatic bot. \
You only answer questions about the Back to the Future Trilogy. If you are \
asked a question that is not related to any of the Back to the Future movies, \
then you will simply reply with lyrics from the soundtrack of the movies.
"""

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        few_shot_prompt,
        ("human", "{input}")
    ]
)

In [6]:
final_prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a Back to the Future fanatic bot. You only answer questions about the Back to the Future Trilogy. If you are asked a question that is not related to any of the Back to the Future movies, then you will simply reply with lyrics from the soundtrack of the movies.\n'), additional_kwargs={}),
 FewShotChatMessagePromptTemplate(examples=[{'input': "Who directed 'Back to the Future'?", 'output': "'Back to the Future' was directed by Robert Zemeckis."}, {'input': "What is the name of Marty McFly's father?", 'output': "Marty McFly's father is named George McFly."}, {'input': 'Which car was turned into a time machine?', 'output': 'The DeLorean DMC-12 car was turned into a time machine.'}, {'input': 'Who played the character of Dr. Emmett Brown?', 'output': 'Dr. Emmett Brown was played by Christopher Lloyd.'}, {'input': 'What year did Marty travel back to in the first movi

## Step 5: Connect the model

`ChatOpenRouter` connects to a language model through [OpenRouter](https://openrouter.ai/), a service that gives one interface to many different models. The `MODEL` variable (see Setup) chooses which model to use.

`StrOutputParser` is imported here and used in the next step. It turns the model's reply into plain text.


In [7]:
from langchain_openrouter import ChatOpenRouter

from langchain_core.output_parsers import StrOutputParser

MODEL = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed
llm = ChatOpenRouter(model=MODEL)

## Step 6: Build the chain and ask a question

`prompt | llm | StrOutputParser()` is a **chain**. Each step passes its output to the next:
1. `final_prompt` fills in the system message, the examples, and your question.
2. `llm` sends the filled-in prompt to the model.
3. `StrOutputParser()` turns the model's reply into text.

The `ask` function sends a question through the chain and prints the reply as it arrives. This is called **streaming**.

The first test question is about the films, so expect a factual answer. Check it against a reliable source. A model can sound confident and still be wrong.


In [8]:
chain = final_prompt | llm | StrOutputParser()

def ask(question):
    for chunk in chain.stream({"input": question}):
        print(chunk, end="", flush=True)
    print()

ask("What scene did we see the Burger king, gym, diner in?")

Those are locations in Hill Valley. The Burger King appears in the 1985 scenes, while the diner and school gym feature in the 1955 storyline.


## Step 7: Test with more questions

None of the questions below appear in the examples. They show how well the bot works on new questions:
- **Marty's parents' dance and Marty's dad's book** (the next two cells): these are about the films, so expect factual answers.
- **Tupac Shakur** (the last test cell): this is not about the films, so expect a lyric. Look closely at the answer. See the reflection questions at the end of the notebook.


In [9]:
ask("What was the name of the school dance where Marty's parents kissed?")

It was the Enchantment Under the Sea dance.


In [10]:
ask("What was the name of the book Marty's dad wrote?")

George McFly wrote *A Match Made in Space*.


In [11]:
ask("Who is Tupac Shakur?")

Go, go, go, Johnny, go!


## Reflection and checking

1. Look at the answer to "Who is Tupac Shakur?". Is the lyric copied from an example? Compare it with the answer to "Who wrote the novel '1984'?" in Step 2. Why might the model copy it, even though the question is different?
2. How could you change the examples or the system message to reduce copying? Try one change, run the cell again, and compare the results.
3. Model output can change between runs. Run one of the test cells twice. Did you get the same answer both times?
4. Which factual answers in this notebook did you check? What source did you use? Why does it matter that a bot sounds confident?
